In [1]:
# imports
import tensorflow as tf
from tensorflow import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt

#tf.keras.utils.set_random_seed(1032)
print(tf.config.list_physical_devices('GPU'))

2024-05-22 02:24:17.447143: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-22 02:24:17.447209: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-22 02:24:17.448033: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-22 02:24:17.453194: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-22 02:24:18.068886: W tensorflow/compiler/tf2

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-05-22 02:24:18.863677: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 02:24:18.884856: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 02:24:18.884900: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

test_data = test_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


Found 2339 files belonging to 10 classes.


2024-05-22 02:24:19.045502: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 02:24:19.045567: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 02:24:19.045587: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 02:24:19.175146: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-22 02:24:19.175200: I external/local_xla/xla/stream_executor

Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS =3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def Model(input_shape):

    model = keras.Sequential([
    keras.layers.Conv2D(32, kernel_size = 3, activation='relu', input_shape=input_shape),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    #keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(64, kernel_size=  3, activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    #keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(96, kernel_size= 3, activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    #keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(96, kernel_size= 3, activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    #keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(64, kernel_size=3 , activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    #keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
        
        
    keras.layers.Flatten(),
    #keras.layers.Dense(256, activation='relu', activity_regularizer = tf.keras.regularizers.L2(0.01)),
    #keras.layers.Dropout(0.6),
    keras.layers.Dense(128, activation='relu', activity_regularizer = tf.keras.regularizers.L2(0.01)),
    keras.layers.Dropout(0.8),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10, activation='softmax', activity_regularizer = tf.keras.regularizers.L2(0.01))
    ])
    
    #, activity_regularizer = tf.keras.regularizers.L2()
    #, activity_regularizer = tf.keras.regularizers.L2(0.008)
    
    return model

Model(input_shape)

In [4]:
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=5e-3,
    decay_steps=12000,
    decay_rate=0.9)


model = Model(input_shape)
model.compile(optimizer = keras.optimizers.Adam(learning_rate = 0.0003),
             loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
             metrics=['accuracy'])

#0.000225

In [5]:
earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=75, restore_best_weights=True)

# min_delta=0.001
#,callbacks=[earlystopping]
history = model.fit(
train_data,
epochs = 400,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping])

Epoch 1/400


2024-05-22 02:24:21.544943: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_1/dropout_6/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2024-05-22 02:24:22.141772: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-05-22 02:24:22.270986: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-05-22 02:24:23.418841: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f4426f0e830 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-22 02:24:23.418890: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2024-05-22 02:24:23.423163: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling ML

74/74 [==============================] - 10s 66ms/step - loss: 2.6951 - accuracy: 0.0953 - val_loss: 2.3047 - val_accuracy: 0.0800
Epoch 2/400
74/74 [==============================] - 4s 50ms/step - loss: 2.5049 - accuracy: 0.1060 - val_loss: 2.3039 - val_accuracy: 0.1200
Epoch 3/400
74/74 [==============================] - 4s 49ms/step - loss: 2.4298 - accuracy: 0.0932 - val_loss: 2.3039 - val_accuracy: 0.1000
Epoch 4/400
74/74 [==============================] - 4s 50ms/step - loss: 2.3773 - accuracy: 0.1167 - val_loss: 2.3079 - val_accuracy: 0.1000
Epoch 5/400
74/74 [==============================] - 4s 48ms/step - loss: 2.3537 - accuracy: 0.1094 - val_loss: 2.3117 - val_accuracy: 0.1000
Epoch 6/400
74/74 [==============================] - 4s 48ms/step - loss: 2.3222 - accuracy: 0.1124 - val_loss: 2.3049 - val_accuracy: 0.1000
Epoch 7/400
74/74 [==============================] - 4s 49ms/step - loss: 2.2999 - accuracy: 0.1231 - val_loss: 2.2972 - val_accuracy: 0.1200
Epoch 8/400
74/74

In [7]:
model.evaluate(val_data, return_dict = True)

2/2 [==============================] - 0s 83ms/step - loss: 0.4706 - accuracy: 0.8400


{'loss': 0.47059738636016846, 'accuracy': 0.8399999737739563}